In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix, roc_curve
import os
import glob
import pandas as pd
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import WeightedRandomSampler

import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.85, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma   
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss) 
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()


class ChestXRayProcessor:
    def __init__(self, data_dir, img_size=512):

        self.data_dir = data_dir
        self.img_size = img_size

    def get_metadata_dataframe(self, split='train'):

        allowed_splits = ['train', 'test', 'val']
        if split not in allowed_splits:
            raise ValueError(f"Split deve essere uno tra {allowed_splits}")

        path_pattern = os.path.join(self.data_dir, split, '*', '*.jpeg')
        file_paths = glob.glob(path_pattern)

        data = []
        for path in file_paths:

            label = os.path.basename(os.path.dirname(path))
            data.append({'image_path': path, 'label': label, 'target': 1 if label == 'PNEUMONIA' else 0})

        df = pd.DataFrame(data)
        return df

    def analyze_dataset(self, df):

        print("--- Analisi Distribuzione delle Classi ---")
        counts = df['label'].value_counts()
        percentages = df['label'].value_counts(normalize=True) * 100
        for idx in counts.index:
            print(f"Classe {idx}: {counts[idx]} immagini ({percentages[idx]:.2f}%)")
        print("------------------------------------------------")

    def get_train_transforms(self):
      cxr_mean = (0.482, 0.482, 0.482)
      cxr_std = (0.236, 0.236, 0.236)

      return A.Compose([
          A.LongestMaxSize(max_size=self.img_size, interpolation=cv2.INTER_AREA),
          A.PadIfNeeded(min_height=self.img_size, min_width=self.img_size, border_mode=cv2.BORDER_REFLECT_101),
          A.Affine(translate_percent={"x": (-0.08, 0.08), "y": (-0.08, 0.08)},
                  scale=(0.92, 1.08), rotate=(-5, 5), border_mode=cv2.BORDER_REFLECT_101, p=0.7),

          A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
          A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
          # -----------------------------------------------------

          A.Normalize(mean=cxr_mean, std=cxr_std),
          ToTensorV2()
      ])

    def get_val_test_transforms(self): #validation e test

        cxr_mean = (0.482, 0.482, 0.482)
        cxr_std = (0.236, 0.236, 0.236)

        return A.Compose([
            A.LongestMaxSize(max_size=self.img_size, interpolation=cv2.INTER_AREA),
            A.PadIfNeeded(min_height=self.img_size, min_width=self.img_size,
                          border_mode=cv2.BORDER_REFLECT_101
                          ),
            #A.CLAHE(clip_limit=1.5, tile_grid_size=(8, 8), p=0.0),

            A.Normalize(mean=cxr_mean, std=cxr_std),
            ToTensorV2()
        ])

class ChestXRayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        target = row['target']

        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(target, dtype=torch.float32)

def prepare_data(data_dir, batch_size=16, img_size=512):
    processor = ChestXRayProcessor(data_dir=data_dir, img_size=img_size)

    train_df = processor.get_metadata_dataframe(split='train')
    test_df = processor.get_metadata_dataframe(split='test')

    try:
        val_df = processor.get_metadata_dataframe(split='val')
    except Exception:
        val_df = pd.DataFrame()

    if val_df.empty:
        print("\n Cartella 'val' vuota o non trovata. Creazione del Validation Set dal Training Set (80/20)...")
        train_df, val_df = train_test_split(
            train_df, test_size=0.2, random_state=42, stratify=train_df['target']
        )

    processor.analyze_dataset(train_df)

    train_dataset = ChestXRayDataset(train_df, transform=processor.get_train_transforms())
    val_dataset = ChestXRayDataset(val_df, transform=processor.get_val_test_transforms())
    test_dataset = ChestXRayDataset(test_df, transform=processor.get_val_test_transforms())

    class_counts = train_df['target'].value_counts().sort_index()
    num_normal = class_counts[0]
    num_pneumonia = class_counts[1]


    class_weights = [1.0 / np.sqrt(num_normal)* 0.85, 1.0 / np.sqrt(num_pneumonia)]

    sample_weights = [class_weights[target] for target in train_df['target'].values]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)


    pos_weight = torch.tensor([num_normal / num_pneumonia], dtype=torch.float32)
    return train_loader, val_loader, test_loader, pos_weight


def get_model():
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

    for module in model.modules():
        if hasattr(module, "memory_efficient"):
            module.memory_efficient = True

    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_ftrs, 1)
    )

    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False
        else:
            param.requires_grad = True

    return model


def train_model(model, train_loader, val_loader, pos_weight, epochs=10, freeze_epochs=3, device='cuda', save_path='best_model.pth'):
    model = model.to(device)

    criterion = FocalLoss(gamma=2.0)

    early_params = []
    late_params = []
    classifier_params = []

    for name, param in model.named_parameters():
        if "classifier" in name:
            classifier_params.append(param)
        #primi blocchi di DenseNet (fino al denseblock2)
        elif any(x in name for x in ["conv0", "norm0", "denseblock1", "transition1", "denseblock2", "transition2"]):
            early_params.append(param)
        #blocchi più profondi (denseblock3 e 4)
        else:
            late_params.append(param)

    #i primi due partono a LR 0.0 perché congelati
    optimizer = optim.AdamW([
        {'params': early_params, 'lr': 3e-6},      #gruppo 0 
        {'params': late_params, 'lr': 7e-6},       #gruppo 1 
        {'params': classifier_params, 'lr': 4e-4}  #gruppo 2 
    ], weight_decay=2e-2)
    # =========================================================================

    best_val_loss = float('inf')
    scaler = torch.amp.GradScaler('cuda')

    backbone_unfrozen = False

    for epoch in range(epochs):

        if not backbone_unfrozen and epoch >= freeze_epochs:

            #gradienti per tutto il backbone
            for param in early_params + late_params:
                param.requires_grad = True

            optimizer.param_groups[0]['lr'] = 3e-6
            optimizer.param_groups[1]['lr'] = 7e-6

            backbone_unfrozen = True

        model.train()
        running_loss = 0.0

        train_bar = tqdm(train_loader, desc=f"Epoca [{epoch+1}/{epochs}] - Training", unit="batch")

        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)

            optimizer.zero_grad()

            with torch.amp.autocast(device_type=('cuda' if 'cuda' in device else 'cpu'), dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        epoch_train_loss = running_loss / len(train_loader.dataset)

        #validazione
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        val_bar = tqdm(val_loader, desc=f"Epoca [{epoch+1}/{epochs}] - Validazione", unit="batch", leave=False)

        with torch.no_grad():
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device).unsqueeze(1)

                with torch.amp.autocast(device_type=('cuda' if 'cuda' in device else 'cpu'), dtype=torch.float16):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).int()

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        epoch_val_loss = val_loss / len(val_loader.dataset)
        val_accuracy = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds)
        val_recall = recall_score(all_labels, all_preds)

        #==========================================

        try:
            tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        except ValueError:

            cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
            tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        #==========================================

        current_lrs = [param_group['lr'] for param_group in optimizer.param_groups]

        print(f"\n [EPOCH {epoch+1} COMPLETATA]")
        print(f"   ↳ Train Loss: {epoch_train_loss:.4f}")
        print(f"   ↳ Val Loss:   {epoch_val_loss:.4f}")
        print(f"   ↳ Val Acc:    {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
        print(f"   ↳ Val F1:     {val_f1:.4f}")
        print(f"   ↳ Val Recall: {val_recall:.4f} (Sensibilità)")
        print(f"   ↳ Val Spec:   {specificity:.4f} (Specificità)")
        print(f"   ↳ Current LR: Early={current_lrs[0]:.2e}, Late={current_lrs[1]:.2e}, Class={current_lrs[2]:.2e}")


        #CHECKPOINTING
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), save_path)
            print(f"   ↳ Modello salvato (Miglior Val Loss)")
        print("-" * 50 + "\n")


#DATA_DIRECTORY = "./chest-xray-pneumonia/chest_xray/chest_xray"
DATA_DIRECTORY = "/content/cartella_XRay/chest_xray"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

train_loader, val_loader, test_loader, pos_weight = prepare_data(
    DATA_DIRECTORY,
    batch_size=16,
    img_size=512
)
densenet = get_model()
train_model(densenet, train_loader, val_loader, pos_weight, epochs=7, freeze_epochs=3, device=DEVICE);

densenet.load_state_dict(torch.load('best_model.pth'))
densenet.eval()



 Cartella 'val' vuota o non trovata. Creazione del Validation Set dal Training Set (80/20)...
--- Analisi Distribuzione delle Classi ---
Classe PNEUMONIA: 3106 immagini (74.22%)
Classe NORMAL: 1079 immagini (25.78%)
------------------------------------------------


Epoca [1/7] - Training: 100%|██████████| 262/262 [02:07<00:00,  2.06batch/s, loss=0.0940]



 [EPOCH 1 COMPLETATA]
   ↳ Train Loss: 0.1151
   ↳ Val Loss:   0.0624
   ↳ Val Acc:    0.9284 (92.84%)
   ↳ Val F1:     0.9513
   ↳ Val Recall: 0.9434 (Sensibilità)
   ↳ Val Spec:   0.8852 (Specificità)
   ↳ Current LR: Early=3.00e-06, Late=7.00e-06, Class=4.00e-04
   ↳ Modello salvato (Miglior Val Loss)
--------------------------------------------------



Epoca [2/7] - Training: 100%|██████████| 262/262 [02:07<00:00,  2.06batch/s, loss=0.1563]



 [EPOCH 2 COMPLETATA]
   ↳ Train Loss: 0.0894
   ↳ Val Loss:   0.0467
   ↳ Val Acc:    0.9436 (94.36%)
   ↳ Val F1:     0.9619
   ↳ Val Recall: 0.9575 (Sensibilità)
   ↳ Val Spec:   0.9037 (Specificità)
   ↳ Current LR: Early=3.00e-06, Late=7.00e-06, Class=4.00e-04
   ↳ Modello salvato (Miglior Val Loss)
--------------------------------------------------



Epoca [3/7] - Training: 100%|██████████| 262/262 [02:02<00:00,  2.14batch/s, loss=0.0475]


... (6 outputs left out)


 [EPOCH 6 COMPLETATA]
   ↳ Train Loss: 0.0278
   ↳ Val Loss:   0.0107
   ↳ Val Acc:    0.9819 (98.19%)
   ↳ Val F1:     0.9877
   ↳ Val Recall: 0.9794 (Sensibilità)
   ↳ Val Spec:   0.9889 (Specificità)
   ↳ Current LR: Early=3.00e-06, Late=7.00e-06, Class=4.00e-04
   ↳ Modello salvato (Miglior Val Loss)
--------------------------------------------------



Epoca [7/7] - Training: 100%|██████████| 262/262 [02:45<00:00,  1.59batch/s, loss=0.0062]



 [EPOCH 7 COMPLETATA]
   ↳ Train Loss: 0.0220
   ↳ Val Loss:   0.0098
   ↳ Val Acc:    0.9828 (98.28%)
   ↳ Val F1:     0.9883
   ↳ Val Recall: 0.9794 (Sensibilità)
   ↳ Val Spec:   0.9926 (Specificità)
   ↳ Current LR: Early=3.00e-06, Late=7.00e-06, Class=4.00e-04
   ↳ Modello salvato (Miglior Val Loss)
--------------------------------------------------



DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [ ]:
!pip install -U albumentations
!pip install opendatasets

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q /content/drive/MyDrive/ChestXRay2017.zip -d /content/cartella_XRay

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix
from tqdm import tqdm


def get_model():
    model = models.densenet121(weights=None) 
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(num_ftrs, 1)
    )
    return model


class ChestXRayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        target = row['target']

        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(target, dtype=torch.float32)

def get_test_transforms(img_size=512):
    cxr_mean = (0.482, 0.482, 0.482)
    cxr_std = (0.236, 0.236, 0.236)
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=cxr_mean, std=cxr_std),
        ToTensorV2()
    ])

def load_test_loader(data_dir, batch_size=16, img_size=512):
    path_pattern = os.path.join(data_dir, 'test', '*', '*.jpeg')
    file_paths = glob.glob(path_pattern)

    data = []
    for path in file_paths:
        label = os.path.basename(os.path.dirname(path))
        data.append({'image_path': path, 'label': label, 'target': 1 if label == 'PNEUMONIA' else 0})

    test_df = pd.DataFrame(data)
    print(f" Immagini di Test trovate nel percorso: {len(test_df)}")

    test_dataset = ChestXRayDataset(test_df, transform=get_test_transforms(img_size))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
                             #, persistant_workers=True
                             )
    return test_loader


def print_metrics(all_labels, all_probs, threshold):
    preds = (all_probs >= threshold).astype(int)

    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds)
    sens = recall_score(all_labels, preds)
    roc_auc = roc_auc_score(all_labels, all_probs)

    tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print("-" * 60)
    print(f" RISULTATI SUL TEST SET (Soglia applicata: {threshold:.4f})")
    print("-" * 60)
    print(f"ROC-AUC:               {roc_auc:.4f} ")
    print(f"Sensibilità (Recall):  {sens:.4f}  ")
    print(f"Specificità:           {spec:.4f}  ")
    print(f"Accuratezza globale:   {acc:.4f}")
    print(f"F1-Score:              {f1:.4f}")
    print("\nMATRICE DI CONFUSIONE:")
    print(f"                  Predetto SANO (0) | Predetto POLMONITE (1)")
    print(f"Reale SANO (0)    | TN: {tn:<11} | FP: {fp:<18} ")
    print(f"Reale POLMONITE(1)| FN: {fn:<11} | TP: {tp:<18} ")
    print("-" * 60 + "\n")

if __name__ == "__main__":

    PATH_PESI = '/content/best_model.pth'
    DATA_DIRECTORY = "/content/cartella_XRay/chest_xray"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    test_loader = load_test_loader(DATA_DIRECTORY)

    densenet = get_model().to(DEVICE)
    if os.path.exists(PATH_PESI):
        densenet.load_state_dict(torch.load(PATH_PESI, map_location=DEVICE))

    densenet.eval()

    all_probs = []
    all_labels = []


    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Inferenza", leave=False):
            images = images.to(DEVICE)

            with torch.amp.autocast(device_type=('cuda' if 'cuda' in DEVICE else 'cpu'), dtype=torch.float16):
                outputs = densenet(images)

            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    soglie_da_testare = [0.4000, 0.5000, 0.5757, 0.7000, 0.8000]

    for soglia in soglie_da_testare:
        print_metrics(all_labels, all_probs, threshold=soglia)

 Immagini di Test trovate nel percorso: 624


------------------------------------------------------------
 RISULTATI SUL TEST SET (Soglia applicata: 0.4000)
------------------------------------------------------------
ROC-AUC:               0.9607 
Sensibilità (Recall):  0.9949  
Specificità:           0.5940  
Accuratezza globale:   0.8446
F1-Score:              0.8889

MATRICE DI CONFUSIONE:
                  Predetto SANO (0) | Predetto POLMONITE (1)
Reale SANO (0)    | TN: 139         | FP: 95                 
Reale POLMONITE(1)| FN: 2           | TP: 388                
------------------------------------------------------------

------------------------------------------------------------
 RISULTATI SUL TEST SET (Soglia applicata: 0.5000)
------------------------------------------------------------
ROC-AUC:               0.9607 
Sensibilità (Recall):  0.9872  
Specificità:           0.7051  
Accuratezza globale:   0.8814
F1-Score:              0.9123

MATRICE DI CONFUSIONE:
                  Predetto SANO (0) | Predetto PO